# Index Tracking with scipy.optimize

This python notebook will show two examples of solving an index tracking problem using scipy.optimize instead of Gurobi.

The code will:

1. Install all requirements to run the code
1. Import data from yahoo finance for a N number of stocks for index SP100
1. Clean the data:
  - Filter tickers with low number of observations;
  - Calculate returns and reformat to wide (dates as rows, tickers as columns).
  - Separate the data intro training (used for optimization) and testing (used for analyzing results).
3. Set up the optimization problem using scipy.optimize and solve it:
  - Unconstrained model
  - Asset constrained model
4. Analyze results in testing data


## Setting up environment

In [ ]:
!pip install scipy pandas numpy pyarrow fastparquet yfinance sklearn matplotlib sktime

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

## Downloading data

In [ ]:
# SP100 tickers list
tickers_list = ['AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'AIG', 'AMD', 'AMGN', 'AMT', 'AMZN',
                'AVGO', 'AXP', 'BA', 'BAC', 'BK', 'BKNG', 'BLK', 'BMY', 'BRK-B', 'C',
                'CAT', 'CHTR', 'CL', 'CMCSA', 'COF', 'COP', 'COST', 'CRM', 'CSCO', 'CVS',
                'CVX', 'DE', 'DHR', 'DIS', 'DOW', 'DUK', 'EMR', 'EXC', 'F', 'FDX',
                'GD', 'GE', 'GILD', 'GM', 'GOOG', 'GOOGL', 'GS', 'HD', 'HON', 'IBM',
                'INTC', 'JNJ', 'JPM', 'KHC', 'KO', 'LIN', 'LLY', 'LMT', 'LOW', 'MA',
                'MCD', 'MDLZ', 'MDT', 'MET', 'META', 'MMM', 'MO', 'MRK', 'MS', 'MSFT',
                'NEE', 'NFLX', 'NKE', 'NVDA', 'ORCL', 'PEP', 'PFE', 'PG', 'PM', 'PYPL',
                'QCOM', 'RTX', 'SBUX', 'SCHW', 'SO', 'SPG', 'T', 'TGT', 'TMO', 'TSLA',
                'TXN', 'UNH', 'UNP', 'UPS', 'USB', 'V', 'VZ', 'WFC', 'WMT', 'XOM']

mkt_index = '^OEX'  # S&P 100 index

# Download data
end_date = datetime.now()
start_date = end_date - timedelta(days=730)  # 2 years of data

print(f"Downloading data from {start_date.date()} to {end_date.date()}...")

# Download stock data
data = yf.download(tickers_list, start=start_date, end=end_date, progress=False)['Adj Close']
# Download market index
mkt_data = yf.download(mkt_index, start=start_date, end=end_date, progress=False)['Adj Close']

print(f"Downloaded {len(data)} observations for {len(data.columns)} tickers")

## Data Cleaning

In [ ]:
# Remove tickers with too many missing values
min_observations = int(0.95 * len(data))
valid_tickers = data.columns[data.count() >= min_observations].tolist()
data = data[valid_tickers]

# Fill any remaining missing values
data = data.fillna(method='ffill').fillna(method='bfill')
mkt_data = mkt_data.fillna(method='ffill').fillna(method='bfill')

# Calculate returns
returns = data.pct_change().dropna()
mkt_returns = mkt_data.pct_change().dropna()

# Align dates
common_dates = returns.index.intersection(mkt_returns.index)
returns = returns.loc[common_dates]
mkt_returns = mkt_returns.loc[common_dates]

print(f"After cleaning: {len(returns)} observations, {len(returns.columns)} tickers")

## Train/Test Split

In [ ]:
# Split data: 70% training, 30% testing
split_idx = int(0.7 * len(returns))

r_train = returns.iloc[:split_idx]
r_test = returns.iloc[split_idx:]
r_mkt_train = mkt_returns.iloc[:split_idx]
r_mkt_test = mkt_returns.iloc[split_idx:]

sampled_tickers = list(returns.columns)
n_assets = len(sampled_tickers)

print(f"Training set: {len(r_train)} observations")
print(f"Test set: {len(r_test)} observations")
print(f"Number of assets: {n_assets}")

## Model 1: Unconstrained Index Tracking

Minimize tracking error: $\min_w \sum_{t=1}^T (r_{p,t} - r_{m,t})^2$

where:
- $w$ = portfolio weights
- $r_{p,t} = \sum_{i=1}^n w_i r_{i,t}$ = portfolio return at time t
- $r_{m,t}$ = market return at time t

In [ ]:
# Define objective function: minimize tracking error
def tracking_error(weights, returns, market_returns):
    """Calculate sum of squared tracking errors"""
    portfolio_returns = returns @ weights
    errors = portfolio_returns - market_returns
    return np.sum(errors ** 2)

# Constraint: weights sum to 1
def weight_constraint(weights):
    return np.sum(weights) - 1.0

# Initial guess: equal weights
w0 = np.ones(n_assets) / n_assets

# Bounds: weights between 0 and 1
bounds = [(0, 1) for _ in range(n_assets)]

# Constraints dict for scipy
constraints = {'type': 'eq', 'fun': weight_constraint}

# Solve
print(f"Optimizing unconstrained model with {n_assets} assets...")
result_unc = minimize(
    tracking_error,
    w0,
    args=(r_train.values, r_mkt_train.values),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000}
)

w_hat_unc = result_unc.x
print(f"Optimization {'successful' if result_unc.success else 'failed'}")
print(f"Number of non-zero weights: {np.sum(w_hat_unc > 1e-6)}")
print(f"Tracking error: {np.sqrt(result_unc.fun / len(r_train)):.6f}")

## Model 2: Asset-Constrained Model

Limit the number of assets in the portfolio (cardinality constraint).

Minimize: $\sum_{t=1}^T (r_{p,t} - r_{m,t})^2$

Subject to:
- $\sum_{i=1}^n w_i = 1$
- $\sum_{i=1}^n I(w_i > 0) \leq K$ (at most K assets)
- $w_i \geq 0$

In [ ]:
# For asset-constrained model, we'll use a greedy approach:
# 1. Start with unconstrained solution
# 2. Select K assets with largest weights
# 3. Re-optimize with only those K assets

K = 10  # Number of assets to select

# Select top K assets by weight
top_k_idx = np.argsort(w_hat_unc)[-K:]
selected_tickers = [sampled_tickers[i] for i in top_k_idx]

print(f"Selected {K} tickers: {selected_tickers}")

# Re-optimize with only selected assets
r_train_selected = r_train[selected_tickers].values
w0_const = np.ones(K) / K
bounds_const = [(0, 1) for _ in range(K)]

def weight_constraint_K(weights):
    return np.sum(weights) - 1.0

constraints_const = {'type': 'eq', 'fun': weight_constraint_K}

print(f"Optimizing asset-constrained model with {K} assets...")
result_const = minimize(
    tracking_error,
    w0_const,
    args=(r_train_selected, r_mkt_train.values),
    method='SLSQP',
    bounds=bounds_const,
    constraints=constraints_const,
    options={'maxiter': 1000}
)

# Create full weight vector
w_hat_const_full = np.zeros(n_assets)
for i, idx in enumerate(top_k_idx):
    w_hat_const_full[idx] = result_const.x[i]

print(f"Optimization {'successful' if result_const.success else 'failed'}")
print(f"Tracking error: {np.sqrt(result_const.fun / len(r_train)):.6f}")

## Results Analysis

In [ ]:
# Calculate portfolio returns on test set
r_hat_unc = r_test.values @ w_hat_unc
r_hat_const = r_test.values @ w_hat_const_full

# Calculate tracking errors
error_unc = r_hat_unc - r_mkt_test.values
error_const = r_hat_const - r_mkt_test.values

# Calculate cumulative returns
cumret_r_unc = np.cumprod(1 + r_hat_unc)
cumret_r_const = np.cumprod(1 + r_hat_const)
cumret_mkt = np.cumprod(1 + r_mkt_test.values)

# Plot results
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(r_test.index, cumret_mkt, label=mkt_index, linewidth=2)
ax.plot(r_test.index, cumret_r_unc, label=f"Unconstrained Model ({n_assets} assets)", alpha=0.8)
ax.plot(r_test.index, cumret_r_const, label=f"Asset Constrained Model ({K} assets)", alpha=0.8)

ax.legend()
ax.set_title("Index Tracking Performance")
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Returns')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print statistics
print("\nTracking Errors (Test Set):")
print(f"  Unconstrained Model ({n_assets} assets): {error_unc.std()*100:.3f}%")
print(f"  Asset Constrained Model ({K} assets): {error_const.std()*100:.3f}%")

print("\nMax Error:")
print(f"  Unconstrained Model ({n_assets} assets): {error_unc.max()*100:.3f}%")
print(f"  Asset Constrained Model ({K} assets): {error_const.max()*100:.3f}%")

print("\nMin Error:")
print(f"  Unconstrained Model ({n_assets} assets): {error_unc.min()*100:.3f}%")
print(f"  Asset Constrained Model ({K} assets): {error_const.min()*100:.3f}%")